# 01 — Explore the FTW benchmark catalog

**Purpose:** decide which collections to use and check that the files hold what we need, before downloading anything large.

The FTW benchmark has one collection per country and year. Each collection has three files we need, joined on `id`: `fields` (crop labels), `field_embeddings` (AEF and OlmoEarth stats per field) and `chips` (2 km squares carrying the train/val/test split). This notebook reads only file footers and a few small columns.

**Output:** the list of collections (`COLLECTIONS`) for notebook 02.

In [16]:
import json
import fsspec
import pandas as pd
import pyarrow.parquet as pq
from urllib.parse import urljoin
from urllib.request import urlopen
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor

In [17]:
BENCHROOT  = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
YEAR       = 2024
COLLECTION = 'ee/ee_2024'
MINFIELDS  = 100
NWORKERS   = 16
FS         = fsspec.filesystem('https')
pd.set_option('display.max_rows',200)

In [18]:
def get_json(url):
    with urlopen(url,timeout=60) as response:
        return json.load(response)

def child_urls(url):
    return [urljoin(url,link['href']) for link in get_json(url)['links'] if link['rel']=='child']

def asset_url(collection,key):
    return urljoin(BENCHROOT,f"{collection}/{collection.split('/')[1]}_{key}.parquet")

def parquet_meta(url):
    with FS.open(url,block_size=2**20) as f:
        pfile = pq.ParquetFile(f)
        return pfile.schema_arrow.names,pfile.metadata.num_rows

def read_columns(url,columns):
    with FS.open(url,block_size=2**22) as f:
        return pq.read_table(f,columns=columns).to_pandas()

## 1. Which collections to use

List every benchmark collection, keep those from `YEAR`, and check which have HCAT crop labels. For each labelled one, read only the `hcat:code` column to count fields, the share with a label, and how many HCAT classes reach `MINFIELDS`. This takes about 2 minutes.

In [19]:
with ThreadPoolExecutor(NWORKERS) as executor:
    collurls = [url for urls in executor.map(child_urls,child_urls(BENCHROOT)) for url in urls]
allcollections = ['/'.join(url.split('/')[-3:-1]) for url in collurls]
yearcollections = [collection for collection in allcollections if collection.endswith(str(YEAR))]
print(f'{len(allcollections)} benchmark collections in total; {len(yearcollections)} are from {YEAR}')

180 benchmark collections in total; 23 are from 2024


In [ ]:
def label_check(collection):
    columns,nrows = parquet_meta(asset_url(collection,'fields'))
    if 'hcat:code' not in columns:
        return dict(collection=collection,nfields=nrows,haslabels=False)
    counts = read_columns(asset_url(collection,'fields'),['hcat:code'])['hcat:code'].value_counts()
    return dict(collection=collection,nfields=nrows,haslabels=True,pctlabelled=100*counts.sum()/nrows,
                nclasses=len(counts),nkept=(counts>=MINFIELDS).sum(),embedmb=FS.size(asset_url(collection,'field_embeddings'))/1e6)

with ThreadPoolExecutor(NWORKERS) as executor:
    checks = pd.DataFrame(list(executor.map(label_check,yearcollections))).sort_values('nfields').reset_index(drop=True)
labelled    = checks[checks.haslabels].drop(columns='haslabels').astype({'nclasses':int,'nkept':int}).reset_index(drop=True)
COLLECTIONS = labelled.collection.tolist()
print(f"{len(labelled)} of {len(checks)} {YEAR} collections have HCAT labels; without: {', '.join(checks.loc[~checks.haslabels,'collection'])}")
print(f'They hold {labelled.nfields.sum():,} fields and {labelled.embedmb.sum()/1e3:,.1f} GB of embeddings')
display(labelled.style.format({'nfields':'{:,}','pctlabelled':'{:.1f}','embedmb':'{:,.0f}'}).hide(axis='index'))
print(f'COLLECTIONS = {COLLECTIONS}')

`nkept` is the number of HCAT classes with at least `MINFIELDS` fields. Four collections have very coarse labels (hr, lt, es_vc, es_cm have 7–21 classes), so they are a poor match for the leave-one-country-out test.

## 2. What the files hold (Estonia pilot)

The column names and sizes of the three files for `COLLECTION`.

In [ ]:
for key in ('fields','field_embeddings','chips'):
    columns,nrows = parquet_meta(asset_url(COLLECTION,key))
    print(f"{key}: {nrows:,} rows, {FS.size(asset_url(COLLECTION,key))/1e6:,.0f} MB\n    {', '.join(columns)}")

**The columns we use:**

| Need | File | Column(s) |
|---|---|---|
| Join key | all three | `id` |
| Crop label | `fields` | `hcat:code` and `hcat:name` (the HCAT class) |
| AEF features | `field_embeddings` | `aef_mean`, `aef_std` (64 numbers each) |
| OlmoEarth features | `field_embeddings` | `olmoearth_mean`, `olmoearth_std` (64 numbers each) |
| Location | `fields` | `geometry` (already lon/lat, EPSG:4326) |
| Split | `chips` | `split` (train/val/test per 2 km chip) |

`hcat:name_en` looks like a class name but is the national crop name translated into English, so we don't use it as the label. There is no year column; the year is in the collection id.

## 3. Estonia checks

Confirm that labels and embeddings line up one to one, count the crops, and look at the split.

In [ ]:
fields  = read_columns(asset_url(COLLECTION,'fields'),['id','hcat:code','hcat:name','hcat:name_en'])
embed   = read_columns(asset_url(COLLECTION,'field_embeddings'),['id','pixel_rule'])
nfields = len(fields)
print(f'{nfields:,} fields and {len(embed):,} embedding rows; {fields.id.isin(embed.id).sum():,} ids match, ids unique: {fields.id.is_unique}')
print(f"{fields['hcat:code'].isna().sum():,} fields have no label; {(embed.pixel_rule=='touched').sum():,} are too small to hold a pixel centre (pixel_rule = 'touched')")

In [ ]:
lookup = (fields.groupby(['hcat:code','hcat:name'])
          .agg(nfields=('id','size'),examplenational=('hcat:name_en','first'))
          .reset_index().sort_values('nfields',ascending=False).reset_index(drop=True))
lookup['pct'] = 100*lookup.nfields/nfields
kept  = lookup[lookup.nfields>=MINFIELDS]
share = kept.nfields.sum()/nfields
print(f'{len(kept)} of {len(lookup)} HCAT classes have ≥ {MINFIELDS} fields, covering {kept.nfields.sum():,} fields ({share:.1%})')
display(kept.style.format({'nfields':'{:,}','pct':'{:.1f}'}).hide(axis='index'))

In [ ]:
chips  = read_columns(asset_url(COLLECTION,'chips'),['id','split'])
counts = chips.split.value_counts().reindex(['train','val','test'])
print(f'{len(chips):,} chips: '+', '.join(f'{split} {n:,} ({n/len(chips):.0%})' for split,n in counts.items()))

The split is spatially blocked: chips are grouped into 6×6 km blocks, and each whole block goes to train, val or test (80/10/10, seed 42). That keeps test fields away from training fields. Fields have no split column; notebook 03 gives each field the split of the chip it overlaps most.

## Summary for notebook 02

Use the collections in `COLLECTIONS`. From each, read `id`, `hcat:code`, `hcat:name` and `geometry` from `fields`, and the AEF and OlmoEarth mean/std from `field_embeddings`, and join them on `id`. Use `hcat:code` / `hcat:name` as the label and drop classes with fewer than `MINFIELDS` fields, fields with no label, and very small fields (including `pixel_rule == 'touched'`). Geometry is already lon/lat, so centroids need no reprojection. One decision is still open: `fallow_land_not_crop` (1,851 fields in Estonia) is not a crop, so we need to choose whether to keep it as a class or drop it.